<a href="https://colab.research.google.com/github/muhammadfaseeh993-beep/unstructuredata_tfidf_repo/blob/main/26i8025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import zipfile
import os

zip_path = '/content/textanalysis_tfidf-main.zip'
extract_path = '/content/textanalysis_tfidf_extracted'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# List the extracted files
for root, dirs, files in os.walk(extract_path):
    for file in files:
        print(os.path.join(root, file))

/content/textanalysis_tfidf_extracted/textanalysis_tfidf-main/requirements.txt
/content/textanalysis_tfidf_extracted/textanalysis_tfidf-main/LICENSE
/content/textanalysis_tfidf_extracted/textanalysis_tfidf-main/test_lab15.py
/content/textanalysis_tfidf_extracted/textanalysis_tfidf-main/student_starter_script.py
/content/textanalysis_tfidf_extracted/textanalysis_tfidf-main/README.md
/content/textanalysis_tfidf_extracted/textanalysis_tfidf-main/.github/workflows/label.yml
/content/textanalysis_tfidf_extracted/textanalysis_tfidf-main/.github/workflows/classroom.yml


In [5]:
base_path = '/content/textanalysis_tfidf_extracted/textanalysis_tfidf-main'

def read_file(filename):
    filepath = os.path.join(base_path, filename)
    with open(filepath, 'r', encoding='utf-8') as f:
        print(f"=== {filename} ===")
        print(f.read())
        print("\n" + "="*40 + "\n")

read_file('README.md')
read_file('student_starter_script.py')
read_file('test_lab15.py')

=== README.md ===
# Unstructured Text Analysis & Automated Triage with TF-IDF

**Course:** Data Science Tools & Techniques  
**Topic:** NLP Pipeline, Bag of Words (BoW), TF-IDF Vectorization, and Unsupervised Topic Discovery  

---

## Scenario: User Triage & Bottleneck Discovery

An online tech and consumer platform has accumulated nearly 4,000 unstructured user posts and support discussions across four domains (*Electronics*, *Mac Hardware*, *Automobiles*, and *Medical/Health*). Human moderators are experiencing reading fatigue, and leadership needs an automated, **100% explainable** pipeline to group incoming posts into topical queues and surface defining vocabulary without relying on expensive black-box models.

In this activity, you will build an end-to-end NLP pipeline that standardizes messy user text, compares **Bag of Words** against **TF-IDF**, discovers hidden topical clusters with **K-Means**, and diagnoses real-world unstructured text pitfalls.

---

## Learning Objectives

In [8]:
import os

base_path = '/content/textanalysis_tfidf_extracted/textanalysis_tfidf-main'

# Let's inspect the files in a way that avoids output truncation.
with open(os.path.join(base_path, 'student_starter_script.py'), 'r', encoding='utf-8') as f:
    starter_content = f.read()

print(f"Length of starter script: {len(starter_content)} characters")
print("=== STARTER SCRIPT CONTENT ===")
print(starter_content)

Length of starter script: 5467 characters
=== STARTER SCRIPT CONTENT ===
"""
Unstructured Text Analysis & Automated Triage with TF-IDF
Course: Data Science Tools & Techniques

Instructions:
Complete each TODO section below. You can run this file interactively cell-by-cell
in VS Code / PyCharm / Spyder (using the `# %%` markers) or directly via terminal:
    python lab15_starter.py
"""


# ## Setup & Dataset Loading

# %%
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

# Download required NLTK resources quietly
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("

In [9]:
with open(os.path.join(base_path, 'test_lab15.py'), 'r', encoding='utf-8') as f:
    test_content = f.read()

print(f"Length of test script: {len(test_content)} characters")
print("=== TEST SCRIPT CONTENT ===")
print(test_content)

Length of test script: 2988 characters
=== TEST SCRIPT CONTENT ===
"""
Automated Pytest Suite for Lab 15: TF-IDF & Unstructured Data Triage
This script validates student implementations in lab15_starter.py.
"""

import os
import pytest
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD

# Import student starter script or check functions
try:
    import lab15_starter as student
except ImportError:
    student = None


def test_starter_file_exists():
    assert os.path.exists("lab15_starter.py"), "lab15_starter.py is missing from the repository."


def test_dataset_loading():
    assert student is not None, "Could not import lab15_starter.py"
    assert hasattr(student, "df"), "DataFrame 'df' is missing."
    assert len(student.df) > 100, "Dataset contains fewer than 100 rows."
    assert "text" in student.df.columns, "Column 'text' is missing

In [13]:
import shutil
import os

# Copy the starter script
shutil.copy('/content/textanalysis_tfidf_extracted/textanalysis_tfidf-main/student_starter_script.py', './lab15_starter.py')

with open('./lab15_starter.py', 'r', encoding='utf-8') as f:
    lines = f.readlines()

print(f"Total lines: {len(lines)}")
for idx in range(45, min(140, len(lines))):
    print(f"{idx+1}: {lines[idx]}", end="")

Total lines: 167
46:     categories=CATEGORIES,
47:     remove=("headers", "footers", "quotes"),
48: )
49: 
50: df = pd.DataFrame(
51:     {
52:         "text": raw_data.data,
53:         "true_label": [raw_data.target_names[i] for i in raw_data.target],
54:     }
55: )
56: 
57: # Keep posts with meaningful length (> 50 characters)
58: df = df[df["text"].str.strip().str.len() > 50].reset_index(drop=True)
59: print(f"Loaded {len(df)} unstructured user posts.\n")
60: 
61: 
62: # %% [markdown]
63: # ## Step 1: Text Cleaning, Tokenization & Normalization (10 mins)
64: 
65: # %%
66: sample_post = df["text"].iloc[0][:400]
67: print("--- Raw Sample Post ---")
68: print(sample_post)
69: print("-" * 40)
70: 
71: # TODO 1.1: Tokenize `sample_post` using `word_tokenize`, convert to lowercase,
72: # and keep only alphabetic tokens (`w.isalpha()`).
73: tokens = None  # Replace None with your list comprehension
74: 
75: # TODO 1.2: Remove standard English stopwords using `stopwords.words('english')`

In [17]:
!python lab15_starter.py

Loading dataset...
Loaded 3703 unstructured user posts.

--- Raw Sample Post ---

My Honda has a cable release that can be locked out with the ignition key.
The valet key can be left with someone and will NOT unlock the trunk
or enable the cable release.
----------------------------------------
Filtered Tokens:   ['honda', 'cable', 'release', 'locked', 'ignition', 'key', 'valet', 'key', 'left', 'someone', 'unlock', 'trunk']
Stemmed Roots:     ['honda', 'cabl', 'releas', 'lock', 'ignit', 'key', 'valet', 'key', 'left', 'someon', 'unlock', 'trunk']
Lemmatized Words:  ['honda', 'cable', 'release', 'lock', 'ignition', 'key', 'valet', 'key', 'leave', 'someone', 'unlock', 'trunk']

TF-IDF Matrix Shape: (3703, 4000) | Sparsity: 99.04%

Top terms per cluster centroid:
Cluster 0: know, like, don, just, does, use, thanks, good
Cluster 1: dsl, cadre, n3jxp, chastity, shameful, surrender, intellect, skepticism
Cluster 2: car, cars, engine, new, just, ford, dealer, like
Cluster 3: mac, apple, drive,

In [18]:
!pytest /content/textanalysis_tfidf_extracted/textanalysis_tfidf-main/test_lab15.py

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0
rootdir: /content
plugins: platformdirs-4.13.5, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.11
collected 6 items                                                              

textanalysis_tfidf_extracted/textanalysis_tfidf-main/test_lab15.py .FFFF [ 83%]
F                                                                        [100%]

=================================== FAILURES ===================================
_____________________________ test_dataset_loading _____________________________

    def test_dataset_loading():
>       assert student is not None, "Could not import lab15_starter.py"
E       AssertionError: Could not import lab15_starter.py
E       assert None is not None

textanalysis_tfidf_extracted/textanalysis_tfidf-main/test_lab15.py:26: AssertionError
___________________________ test_step1_tokenization _______________________

In [25]:
!PYTHONPATH=. pytest /content/textanalysis_tfidf_extracted/textanalysis_tfidf-main/test_lab15.py

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0
rootdir: /content
plugins: platformdirs-4.13.5, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.11
collected 6 items                                                              

textanalysis_tfidf_extracted/textanalysis_tfidf-main/test_lab15.py ..... [ 83%]
.                                                                        [100%]

============================== 6 passed in 11.55s ==============================


In [26]:
!PYTHONPATH=. pytest /content/textanalysis_tfidf_extracted/textanalysis_tfidf-main/test_lab15.py

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0
rootdir: /content
plugins: platformdirs-4.13.5, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.11
collected 6 items                                                              

textanalysis_tfidf_extracted/textanalysis_tfidf-main/test_lab15.py ..... [ 83%]
.                                                                        [100%]

============================== 6 passed in 10.86s ==============================


In [27]:
!PYTHONPATH=. pytest /content/textanalysis_tfidf_extracted/textanalysis_tfidf-main/test_lab15.py

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0
rootdir: /content
plugins: platformdirs-4.13.5, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.11
collected 6 items                                                              

textanalysis_tfidf_extracted/textanalysis_tfidf-main/test_lab15.py ..... [ 83%]
.                                                                        [100%]

============================== 6 passed in 11.46s ==============================


In [28]:
!PYTHONPATH=. pytest /content/textanalysis_tfidf_extracted/textanalysis_tfidf-main/test_lab15.py

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0
rootdir: /content
plugins: platformdirs-4.13.5, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.11
collected 6 items                                                              

textanalysis_tfidf_extracted/textanalysis_tfidf-main/test_lab15.py ..... [ 83%]
.                                                                        [100%]

============================== 6 passed in 15.85s ==============================


In [29]:
!PYTHONPATH=. pytest /content/textanalysis_tfidf_extracted/textanalysis_tfidf-main/test_lab15.py

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0
rootdir: /content
plugins: platformdirs-4.13.5, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.11
collected 6 items                                                              

textanalysis_tfidf_extracted/textanalysis_tfidf-main/test_lab15.py ..... [ 83%]
.                                                                        [100%]

============================== 6 passed in 13.14s ==============================


In [30]:
!PYTHONPATH=. pytest /content/textanalysis_tfidf_extracted/textanalysis_tfidf-main/test_lab15.py

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0
rootdir: /content
plugins: platformdirs-4.13.5, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.11
collected 6 items                                                              

textanalysis_tfidf_extracted/textanalysis_tfidf-main/test_lab15.py ..... [ 83%]
.                                                                        [100%]

============================== 6 passed in 11.76s ==============================


In [31]:
!PYTHONPATH=. pytest /content/textanalysis_tfidf_extracted/textanalysis_tfidf-main/test_lab15.py

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0
rootdir: /content
plugins: platformdirs-4.13.5, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.11
collected 6 items                                                              

textanalysis_tfidf_extracted/textanalysis_tfidf-main/test_lab15.py ..... [ 83%]
.                                                                        [100%]

============================== 6 passed in 11.87s ==============================


In [19]:
%%writefile lab15_starter.py
"""
Unstructured Text Analysis & Automated Triage with TF-IDF
Course: Data Science Tools & Techniques
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

# Download required NLTK resources quietly
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)

# Load 4 real-world unstructured user discussion categories
CATEGORIES = [
    "sci.electronics",
    "rec.autos",
    "comp.sys.mac.hardware",
    "sci.med",
]

print("Loading dataset...")
raw_data = fetch_20newsgroups(
    subset="all",
    categories=CATEGORIES,
    remove=("headers", "footers", "quotes"),
)

df = pd.DataFrame(
    {
        "text": raw_data.data,
        "true_label": [raw_data.target_names[i] for i in raw_data.target],
    }
)

# Keep posts with meaningful length (> 50 characters)
df = df[df["text"].str.strip().str.len() > 50].reset_index(drop=True)
print(f"Loaded {len(df)} unstructured user posts.\n")

# ## Step 1: Text Cleaning, Tokenization & Normalization (10 mins)
sample_post = df["text"].iloc[0][:400]
print("--- Raw Sample Post ---")
print(sample_post)
print("-" * 40)

# TODO 1.1: Tokenize `sample_post` using `word_tokenize`, convert to lowercase, and keep only alphabetic tokens
tokens = [w.lower() for w in word_tokenize(sample_post) if w.isalpha()]

# TODO 1.2: Remove standard English stopwords
stop_words = set(stopwords.words("english"))
filtered_tokens = [w for w in tokens if w not in stop_words]

# TODO 1.3: Apply PorterStemmer and WordNetLemmatizer (with pos='v') to `filtered_tokens`.
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

stemmed_tokens = [stemmer.stem(w) for w in filtered_tokens]
lemmatized_tokens = [lemmatizer.lemmatize(w, pos='v') for w in filtered_tokens]

if filtered_tokens is not None:
    print("Filtered Tokens:  ", filtered_tokens[:12])
    print("Stemmed Roots:    ", stemmed_tokens[:12])
    print("Lemmatized Words: ", lemmatized_tokens[:12])

# ## Step 2: Bag of Words vs. TF-IDF Feature Extraction (15 mins)
# TODO 2.1: Initialize CountVectorizer and TfidfVectorizer
bow_vec = CountVectorizer(stop_words='english', max_features=4000, min_df=5)
tfidf_vec = TfidfVectorizer(stop_words='english', max_features=4000, min_df=5)

# TODO 2.2: Fit and transform `df['text']` to create `X_bow` and `X_tfidf`
X_bow = bow_vec.fit_transform(df['text'])
X_tfidf = tfidf_vec.fit_transform(df['text'])

# TODO 2.3: Calculate matrix sparsity
total_elements = X_tfidf.shape[0] * X_tfidf.shape[1]
sparsity = 100.0 * (1.0 - (X_tfidf.nnz / total_elements))
print(f"\nTF-IDF Matrix Shape: {X_tfidf.shape} | Sparsity: {sparsity:.2f}%")

# TODO 2.4: Compute mean score per feature across documents and plot Top 10 TF-IDF words
tfidf_means = np.asarray(X_tfidf.mean(axis=0)).ravel()
top_idx = tfidf_means.argsort()[-10:]
top_words = tfidf_vec.get_feature_names_out()[top_idx]
top_scores = tfidf_means[top_idx]

plt.figure(figsize=(8, 4.5))
plt.barh(top_words, top_scores, color="teal")
plt.xlabel("Average TF-IDF Weight")
plt.title("Top 10 Most Influential Words (Baseline TF-IDF)")
plt.tight_layout()
plt.savefig("top_features.png", dpi=150)
plt.close()

# ## Step 3: Automated Topic Discovery & Spatial Mapping (15 mins)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X_tfidf)

# TODO 3.2: Print the top 8 defining TF-IDF words per cluster centroid and plot heatmap
order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]
terms = tfidf_vec.get_feature_names_out()

cluster_data = {}
print("\nTop terms per cluster centroid:")
for i in range(4):
    top_terms = [terms[ind] for ind in order_centroids[i, :8]]
    print(f"Cluster {i}: {', '.join(top_terms)}")

plt.figure(figsize=(6, 4))
sns.heatmap(pd.DataFrame(kmeans.cluster_centers_[:, order_centroids[:, :5].flatten()]).T, cmap="Blues")
plt.title("Cluster Top Terms Heatmap")
plt.savefig("cluster_heatmap.png", dpi=150)
plt.close()

# TODO 3.3: Project `X_tfidf` to 2 dimensions
svd = TruncatedSVD(n_components=2, random_state=42)
X_2d = svd.fit_transform(X_tfidf)

plt.figure(figsize=(8, 6))
sns.scatterplot(x=X_2d[:, 0], y=X_2d[:, 1], hue=df["cluster"], palette="viridis")
plt.title("Spatial Mapping (TruncatedSVD)")
plt.savefig("spatial_mapping.png", dpi=150)
plt.close()

# ## Step 4: Problem-Solving Diagnostics (20 mins)
# Challenge 1 (Noise Filtering)
stop_custom = list(stopwords.words('english')) + ['know', 'like', 'just', 'think', 'does', 'use', 'thanks']
clean_tfidf_vec = TfidfVectorizer(token_pattern=r'\b[a-zA-Z]{3,}\b', max_df=0.5, stop_words=stop_custom, max_features=4000, min_df=5)
X_clean_tfidf = clean_tfidf_vec.fit_transform(df['text'])

# Challenge 2 (Hardware Overlap Diagnosis)
cross_tab = pd.crosstab(df['true_label'], df['cluster'])
print("\n--- True Labels vs. Clusters ---")
print(cross_tab)

# Challenge 3 (Rant Length Bias)
lengths = df['text'].str.len()
idx_max = lengths.idxmax()
idx_min = lengths.idxmin()

bow_norm_max = np.linalg.norm(X_bow[idx_max].toarray())
bow_norm_min = np.linalg.norm(X_bow[idx_min].toarray())

tfidf_norm_max = np.linalg.norm(X_tfidf[idx_max].toarray())
tfidf_norm_min = np.linalg.norm(X_tfidf[idx_min].toarray())

print(f"\nLongest vs. Shortest Document Norm comparison:")
print(f"BoW - Longest Norm: {bow_norm_max:.2f} | Shortest Norm: {bow_norm_min:.2f}")
print(f"TF-IDF - Longest Norm: {tfidf_norm_max:.2f} | Shortest Norm: {tfidf_norm_min:.2f}")

# Challenge 4 (Stakeholder Explainability Audit)
def explain_document(doc_id, matrix, vectorizer, top_n=5):
    row = matrix[doc_id].toarray().ravel()
    non_zero_indices = row.argsort()[::-1][:top_n]
    feature_names = vectorizer.get_feature_names_out()
    explanation = [(feature_names[i], row[i]) for i in non_zero_indices if row[i] > 0]
    return explanation


Overwriting lab15_starter.py


In [21]:
!pytest /content/textanalysis_tfidf_extracted/textanalysis_tfidf-main/test_lab15.py

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0
rootdir: /content
plugins: platformdirs-4.13.5, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.11
collected 6 items                                                              

textanalysis_tfidf_extracted/textanalysis_tfidf-main/test_lab15.py .FFFF [ 83%]
F                                                                        [100%]

=================================== FAILURES ===================================
_____________________________ test_dataset_loading _____________________________

    def test_dataset_loading():
>       assert student is not None, "Could not import lab15_starter.py"
E       AssertionError: Could not import lab15_starter.py
E       assert None is not None

textanalysis_tfidf_extracted/textanalysis_tfidf-main/test_lab15.py:26: AssertionError
___________________________ test_step1_tokenization _______________________

In [22]:
%%writefile lab15_starter.py
"""
Unstructured Text Analysis & Automated Triage with TF-IDF
Course: Data Science Tools & Techniques
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

# Download required NLTK resources quietly
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)

# Load 4 real-world unstructured user discussion categories
CATEGORIES = [
    "sci.electronics",
    "rec.autos",
    "comp.sys.mac.hardware",
    "sci.med",
]

print("Loading dataset...")
raw_data = fetch_20newsgroups(
    subset="all",
    categories=CATEGORIES,
    remove=("headers", "footers", "quotes"),
)

df = pd.DataFrame(
    {
        "text": raw_data.data,
        "true_label": [raw_data.target_names[i] for i in raw_data.target],
    }
)

# Keep posts with meaningful length (> 50 characters)
df = df[df["text"].str.strip().str.len() > 50].reset_index(drop=True)
print(f"Loaded {len(df)} unstructured user posts.\n")

# ## Step 1: Text Cleaning, Tokenization & Normalization (10 mins)
sample_post = df["text"].iloc[0][:400]
print("--- Raw Sample Post ---")
print(sample_post)
print("-" * 40)

# TODO 1.1: Tokenize `sample_post` using `word_tokenize`, convert to lowercase, and keep only alphabetic tokens (`w.isalpha()`).
tokens = [w.lower() for w in word_tokenize(sample_post) if w.isalpha()]

# TODO 1.2: Remove standard English stopwords using `stopwords.words('english')`.
stop_words = set(stopwords.words("english"))
filtered_tokens = [w for w in tokens if w not in stop_words]

# TODO 1.3: Apply PorterStemmer and WordNetLemmatizer (with pos='v') to `filtered_tokens`.
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

stemmed_tokens = [stemmer.stem(w) for w in filtered_tokens]
lemmatized_tokens = [lemmatizer.lemmatize(w, pos='v') for w in filtered_tokens]

if filtered_tokens is not None:
    print("Filtered Tokens:  ", filtered_tokens[:12])
    print("Stemmed Roots:    ", stemmed_tokens[:12])
    print("Lemmatized Words: ", lemmatized_tokens[:12])

# ## Step 2: Bag of Words vs. TF-IDF Feature Extraction (15 mins)
# TODO 2.1: Initialize CountVectorizer and TfidfVectorizer
bow_vec = CountVectorizer(stop_words='english', max_features=4000, min_df=5)
tfidf_vec = TfidfVectorizer(stop_words='english', max_features=4000, min_df=5)

# TODO 2.2: Fit and transform `df['text']` to create `X_bow` and `X_tfidf`
X_bow = bow_vec.fit_transform(df['text'])
X_tfidf = tfidf_vec.fit_transform(df['text'])

# TODO 2.3: Calculate matrix sparsity
total_elements = X_tfidf.shape[0] * X_tfidf.shape[1]
sparsity = 100.0 * (1.0 - (X_tfidf.nnz / total_elements))
print(f"\nTF-IDF Matrix Shape: {X_tfidf.shape} | Sparsity: {sparsity:.2f}%")

# TODO 2.4: Compute mean score per feature across documents and plot Top 10 TF-IDF words
tfidf_means = np.asarray(X_tfidf.mean(axis=0)).ravel()
top_idx = tfidf_means.argsort()[-10:]
top_words = tfidf_vec.get_feature_names_out()[top_idx]
top_scores = tfidf_means[top_idx]

plt.figure(figsize=(8, 4.5))
plt.barh(top_words, top_scores, color="teal")
plt.xlabel("Average TF-IDF Weight")
plt.title("Top 10 Most Influential Words (Baseline TF-IDF)")
plt.tight_layout()
plt.savefig("top_features.png", dpi=150)
plt.close()

# ## Step 3: Automated Topic Discovery & Spatial Mapping (15 mins)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X_tfidf)

# TODO 3.2: Print the top 8 defining TF-IDF words per cluster centroid and plot heatmap
order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]
terms = tfidf_vec.get_feature_names_out()

cluster_data = {}
print("\nTop terms per cluster centroid:")
for i in range(4):
    top_terms = [terms[ind] for ind in order_centroids[i, :8]]
    print(f"Cluster {i}: {', '.join(top_terms)}")
    # collect top 5 words for visual heatmap representation
    for idx, term in enumerate(top_terms[:5]):
        cluster_data[f"C{i}_{term}"] = kmeans.cluster_centers_[i, tfidf_vec.vocabulary_[term]]

# Plot simple helper representation or heatmap
plt.figure(figsize=(6, 4))
sns.heatmap(pd.DataFrame(kmeans.cluster_centers_[:, order_centroids[:, :5].flatten()]).T, cmap="Blues")
plt.title("Cluster Top Terms Heatmap")
plt.savefig("cluster_heatmap.png", dpi=150)
plt.close()

# TODO 3.3: Project `X_tfidf` to 2 dimensions
svd = TruncatedSVD(n_components=2, random_state=42)
X_2d = svd.fit_transform(X_tfidf)

plt.figure(figsize=(8, 6))
sns.scatterplot(x=X_2d[:, 0], y=X_2d[:, 1], hue=df["cluster"], palette="viridis")
plt.title("Spatial Mapping (TruncatedSVD)")
plt.savefig("spatial_mapping.png", dpi=150)
plt.close()

# ## Step 4: Problem-Solving Diagnostics (20 mins)
# Challenge 1 (Noise Filtering)
stop_custom = list(stopwords.words('english')) + ['know', 'like', 'just', 'think', 'does', 'use', 'thanks']
clean_tfidf_vec = TfidfVectorizer(token_pattern=r'\b[a-zA-Z]{3,}\b', max_df=0.5, stop_words=stop_custom, max_features=4000, min_df=5)
X_clean_tfidf = clean_tfidf_vec.fit_transform(df['text'])

# Challenge 2 (Hardware Overlap Diagnosis)
cross_tab = pd.crosstab(df['true_label'], df['cluster'])
print("\n--- True Labels vs. Clusters ---")
print(cross_tab)

# Challenge 3 (Rant Length Bias)
lengths = df['text'].str.len()
idx_max = lengths.idxmax()
idx_min = lengths.idxmin()

bow_norm_max = np.linalg.norm(X_bow[idx_max].toarray())
bow_norm_min = np.linalg.norm(X_bow[idx_min].toarray())

tfidf_norm_max = np.linalg.norm(X_tfidf[idx_max].toarray())
tfidf_norm_min = np.linalg.norm(X_tfidf[idx_min].toarray())

print(f"\nLongest vs. Shortest Document Norm comparison:")
print(f"BoW - Longest Norm: {bow_norm_max:.2f} | Shortest Norm: {bow_norm_min:.2f}")
print(f"TF-IDF - Longest Norm: {tfidf_norm_max:.2f} | Shortest Norm: {tfidf_norm_min:.2f}")

# Challenge 4 (Stakeholder Explainability Audit)
def explain_document(doc_id, matrix, vectorizer, top_n=5):
    row = matrix[doc_id].toarray().ravel()
    non_zero_indices = row.argsort()[::-1][:top_n]
    feature_names = vectorizer.get_feature_names_out()
    explanation = [(feature_names[i], row[i]) for i in non_zero_indices if row[i] > 0]
    return explanation


Overwriting lab15_starter.py


In [24]:
!pytest /content/textanalysis_tfidf_extracted/textanalysis_tfidf-main/test_lab15.py

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0
rootdir: /content
plugins: platformdirs-4.13.5, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.11
collected 6 items                                                              

textanalysis_tfidf_extracted/textanalysis_tfidf-main/test_lab15.py .FFFF [ 83%]
F                                                                        [100%]

=================================== FAILURES ===================================
_____________________________ test_dataset_loading _____________________________

    def test_dataset_loading():
>       assert student is not None, "Could not import lab15_starter.py"
E       AssertionError: Could not import lab15_starter.py
E       assert None is not None

textanalysis_tfidf_extracted/textanalysis_tfidf-main/test_lab15.py:26: AssertionError
___________________________ test_step1_tokenization _______________________

In [14]:
import shutil

# Let's copy the starter script to 'lab15_starter.py' in the current working directory first so it is importable
shutil.copy('/content/textanalysis_tfidf_extracted/textanalysis_tfidf-main/student_starter_script.py', './lab15_starter.py')

# Let's read and print the middle lines of the file to see Step 1 and Step 2 TODOs
with open('./lab15_starter.py', 'r', encoding='utf-8') as f:
    lines = f.readlines()

print(f"Total lines: {len(lines)}")
# Print lines around Step 1 and Step 2 (typically between lines 50 and 150)
for i, line in enumerate(lines):
    if any(step in line for step in ["Step 1", "Step 2", "TODO"]):
        print(f"L{i+1}: {line.strip()}")

Total lines: 167
L6: Complete each TODO section below. You can run this file interactively cell-by-cell
L63: # ## Step 1: Text Cleaning, Tokenization & Normalization (10 mins)
L71: # TODO 1.1: Tokenize `sample_post` using `word_tokenize`, convert to lowercase,
L75: # TODO 1.2: Remove standard English stopwords using `stopwords.words('english')`.
L79: # TODO 1.3: Apply PorterStemmer and WordNetLemmatizer (with pos='v') to `filtered_tokens`.
L93: # ## Step 2: Bag of Words vs. TF-IDF Feature Extraction (15 mins)
L96: # TODO 2.1: Initialize CountVectorizer and TfidfVectorizer with:
L101: # TODO 2.2: Fit and transform `df['text']` to create `X_bow` and `X_tfidf`
L105: # TODO 2.3: Calculate matrix sparsity (% of zero entries in `X_tfidf`)
L111: # TODO 2.4: Compute mean score per feature across documents and plot Top 10 TF-IDF words
L131: # TODO 3.1: Fit KMeans with `n_clusters=4, random_state=42, n_init=10` on `X_tfidf`
L135: # TODO 3.2: Print the top 8 defining TF-IDF words per cluster cent

In [20]:
# Print lines 45 to 130 to inspect the missing parts
for idx in range(45, min(140, len(lines))):
    print(f"{idx+1}: {lines[idx]}", end="")

46:     categories=CATEGORIES,
47:     remove=("headers", "footers", "quotes"),
48: )
49: 
50: df = pd.DataFrame(
51:     {
52:         "text": raw_data.data,
53:         "true_label": [raw_data.target_names[i] for i in raw_data.target],
54:     }
55: )
56: 
57: # Keep posts with meaningful length (> 50 characters)
58: df = df[df["text"].str.strip().str.len() > 50].reset_index(drop=True)
59: print(f"Loaded {len(df)} unstructured user posts.\n")
60: 
61: 
62: # %% [markdown]
63: # ## Step 1: Text Cleaning, Tokenization & Normalization (10 mins)
64: 
65: # %%
66: sample_post = df["text"].iloc[0][:400]
67: print("--- Raw Sample Post ---")
68: print(sample_post)
69: print("-" * 40)
70: 
71: # TODO 1.1: Tokenize `sample_post` using `word_tokenize`, convert to lowercase,
72: # and keep only alphabetic tokens (`w.isalpha()`).
73: tokens = None  # Replace None with your list comprehension
74: 
75: # TODO 1.2: Remove standard English stopwords using `stopwords.words('english')`.
76: stop_words 